In [47]:
!git clone https://github.com/ashba93/hackathon-sapienza.git

Cloning into 'hackathon-sapienza'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 32 (delta 0), reused 0 (delta 0), pack-reused 25 (from 2)
Receiving objects: 100% (32/32), 61.11 MiB | 21.24 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Updating files: 100% (24/24), done.


In [48]:
import pickle

def load_pickle(filepath):
    try:
        with open(filepath, 'rb') as f:
            payload = pickle.load(f)
        print("Artifact successfully loaded.")
        return payload
    except FileNotFoundError:
        print(f"Could not find the file at {filepath}. Please verify the path.")
        raise

model_payload = load_pickle("/content/hackathon-sapienza/data/model_artifact")

Artifact successfully loaded.


In [49]:
%cd hackathon-sapienza

/content/hackathon-sapienza/hackathon-sapienza


In [50]:
#libraries
import glob
import pandas as pd
from pathlib import Path
import zipfile, shutil
import copy
import glob
import pickle
import time
from google.colab import files
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
import textwrap

In [51]:
DATA_DIR = "/content/hackathon-sapienza/data"
part_files = sorted(glob.glob(f"{DATA_DIR}/*part-*"))

In [52]:
print(f"File number: {len(part_files)}")

File number: 10


In [53]:
train_df = pd.concat([pd.read_csv(f, sep=";") for f in part_files], ignore_index=True)

In [54]:
forget_df = pd.read_csv(f"{DATA_DIR}/forget_data.csv")

In [55]:
model_payload = load_pickle("/content/hackathon-sapienza/data/model_artifact")

Artifact successfully loaded.


In [56]:
print(f"Train data Number: {len(train_df)}")
print(f"Unlearn Data Number: {len(forget_df)}")

Train data Number: 129783
Unlearn Data Number: 9085


**Exploratory analysis**


In [57]:
# -check tabular data columns are the same ---
# sanity check the two now have identical columns
print("same columns:", list(train_df.columns) == list(forget_df.columns))

same columns: True


In [58]:
# --- 1. Basic shape and columns ---
print("train:", train_df.shape, "| forget:", forget_df.shape)
print("\nColumns:", list(train_df.columns))
print("\nDtypes:\n", train_df.dtypes.value_counts())
train_df.head()

train: (129783, 410) | forget: (9085, 410)

Columns: ['ang_m_datediff_att_linea', 'ang_m_flag_mnp', 'ang_m_datediff_cessazione', 'ang_m_datediff_scad_sim', 'ang_m_cns_cre_res_per', 'ang_m_anz_linea', 'ang_m_datediff_mnp_in', 'ang_m_datediff_cam_prof', 'ang_m_datediff_mnp_out', 'ohe_ang_m_stato_linea_a', 'ohe_ang_m_stato_linea_c', 'ohe_ang_m_stato_linea_n', 'ohe_ang_m_stato_linea_s', 'ohe_ang_m_tipo_linea_fwa', 'ohe_ang_m_tipo_linea_pp', 'ohe_ang_m_tipo_linea_xt', 'ohe_ang_m_area_territoriale_altro_nd', 'ohe_ang_m_area_territoriale_c', 'ohe_ang_m_area_territoriale_ne', 'ohe_ang_m_area_territoriale_no', 'ohe_ang_m_area_territoriale_s', 'ohe_ang_m_pdv_canale_des_top_4_altro', 'ohe_ang_m_pdv_canale_des_top_4_grande_distribuzione', 'ohe_ang_m_pdv_canale_des_top_4_null', 'ohe_ang_m_pdv_canale_des_top_4_one_team', 'ohe_ang_m_pdv_canale_des_top_4_retail', 'ohe_ang_m_pdv_canale_des_top_4_tele_sales', 'ohe_ang_m_mnp_olo_prov_infrastrutturati', 'ohe_ang_m_mnp_olo_prov_mvno', 'ohe_ang_m_mnp_olo_pr

,ang_m_datediff_att_linea,ang_m_flag_mnp,ang_m_datediff_cessazione,ang_m_datediff_scad_sim,ang_m_cns_cre_res_per,ang_m_anz_linea,ang_m_datediff_mnp_in,ang_m_datediff_cam_prof,ang_m_datediff_mnp_out,ohe_ang_m_stato_linea_a,...,target__family_and_parenting,target__sports,target__food_and_drink,target__pets,target__performance,target__business,target__campagne,target__roaming,target__assicurazioni,user_id
0,0,1,-1,-11,5.000000,0,-1,-1,-1,0,...,0,0,0,0,0,0,0,0,0,38
1,59,1,-1,-10,4.292490,59,59,-1,-1,1,...,0,0,0,0,0,0,0,0,0,358
2,19,1,-1,-10,75.249308,18,18,-1,-1,1,...,0,0,0,0,0,0,0,0,0,704
3,133,0,-1,-11,1.545433,133,-1,94,-1,1,...,0,0,0,0,0,0,0,0,0,772
4,15,0,-1,-11,0.010000,15,-1,-1,-1,1,...,0,0,0,0,0,0,0,0,0,815


In [59]:
# --- 2. Find the ID column ---
# An ID column should be (near-)unique per row and present in both dataframes
for c in train_df.columns:
    n_unique = train_df[c].nunique()
    if n_unique > 0.9 * len(train_df):
        in_forget = c in forget_df.columns
        print(f"ID candidate: {c!r}  unique={n_unique}/{len(train_df)}  also_in_forget={in_forget}")

ID candidate: 'user_id'  unique=129783/129783  also_in_forget=True


In [60]:
# --- 3. Check forget ⊂ train (it should be, by definition of a forget set) ---
ID_COL = "user_id"   # <- set to whatever step 2 found
overlap = forget_df[ID_COL].isin(train_df[ID_COL]).mean()
print(f"{overlap:.1%} of forget ids are in train")   # expect ~100%
print("duplicated ids in train:", train_df[ID_COL].duplicated().sum())

100.0% of forget ids are in train
duplicated ids in train: 0


In [61]:
# --- 4. Separate features from targets ---
# Binary 0/1 columns are likely the interest labels (multi-label targets)
binary_cols = [c for c in train_df.columns
               if c != ID_COL and train_df[c].dropna().isin([0, 1]).all()]
non_binary  = [c for c in train_df.columns if c not in binary_cols + [ID_COL]]

print(f"{len(binary_cols)} binary cols (target candidates):", binary_cols[:10])
print(f"\n{len(non_binary)} non-binary cols (feature candidates):", non_binary[:10])

# Sanity: how many labels per user on average? P@10 implies multi-label
print("\nLabels per row (if binary_cols are the targets):",
      train_df[binary_cols].sum(axis=1).describe())

184 binary cols (target candidates): ['ang_m_flag_mnp', 'ohe_ang_m_stato_linea_a', 'ohe_ang_m_stato_linea_c', 'ohe_ang_m_stato_linea_n', 'ohe_ang_m_stato_linea_s', 'ohe_ang_m_tipo_linea_fwa', 'ohe_ang_m_tipo_linea_pp', 'ohe_ang_m_tipo_linea_xt', 'ohe_ang_m_area_territoriale_altro_nd', 'ohe_ang_m_area_territoriale_c']

225 non-binary cols (feature candidates): ['ang_m_datediff_att_linea', 'ang_m_datediff_cessazione', 'ang_m_datediff_scad_sim', 'ang_m_cns_cre_res_per', 'ang_m_anz_linea', 'ang_m_datediff_mnp_in', 'ang_m_datediff_cam_prof', 'ang_m_datediff_mnp_out', 'trf_m_p_d_tu_on_tass', 'trf_m_p_n_tu_on_tass']

Labels per row (if binary_cols are the targets): count    129783.000000
mean         27.946780
std          11.163979
min           8.000000
25%          20.000000
50%          32.000000
75%          36.000000
max          61.000000
dtype: float64


In [62]:
# --- 5. Missing values + basic distributions ---
missing = train_df.isna().mean().sort_values(ascending=False)
print("Cols with missing values:\n", missing[missing > 0])

# Compare forget vs retain distributions on a few features —
# tells you whether the forget set is a random sample or a biased slice
retain_df = train_df[~train_df[ID_COL].isin(set(forget_df[ID_COL]))]
for c in non_binary[:5]:
    print(f"{c}: retain mean={retain_df[c].mean():.3f}  forget mean={forget_df[c].mean():.3f}")

Cols with missing values:
 tac                              0.168242
usg_m_delta_user_dati            0.136366
usg_m_fascia_usg_dati_encoded    0.084110
dtype: float64
ang_m_datediff_att_linea: retain mean=84.497  forget mean=83.536
ang_m_datediff_cessazione: retain mean=-0.635  forget mean=-0.674
ang_m_datediff_scad_sim: retain mean=-14.202  forget mean=-14.262
ang_m_cns_cre_res_per: retain mean=8.710  forget mean=8.615
ang_m_anz_linea: retain mean=86.394  forget mean=85.736


In [63]:
ID_COL = "user_id"
TARGET_COLS = [c for c in train_df.columns if c.startswith("target__")]
FEATURE_COLS = [c for c in train_df.columns if c not in TARGET_COLS + [ID_COL]]

print(len(TARGET_COLS), "targets |", len(FEATURE_COLS), "features")
assert len(TARGET_COLS) == 28 and len(FEATURE_COLS) == 381
assert None not in FEATURE_COLS




28 targets | 381 features


In [64]:
#model
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

state_dict = model_payload["state_dict"]
architecture = model_payload["architecture"]
best_params = model_payload["best_hyperparameters"]
model_class_source = model_payload["model_class_source"]

namespace = {"torch": torch, "nn": nn, "F": F}
exec(textwrap.dedent(model_class_source), namespace)
ModelClass = namespace["DynamicMLP"]

model = ModelClass(**architecture)
model.load_state_dict(state_dict)
model.to(DEVICE).eval()
print("model rebuilt:", sum(p.numel() for p in model.parameters()), "params")

model rebuilt: 45544 params


In [65]:
# --- 6. Cross-check against the model's expected input size ---

print("architecture:", architecture)
first_key = next(iter(state_dict))
print("first layer weight shape:", state_dict[first_key].shape)

architecture: {'input_dim': 381, 'hidden_layers': [85, 111], 'num_outputs': 28}
first layer weight shape: torch.Size([85, 381])


In [66]:
#Split and tensors

forget_ids = set(forget_df[ID_COL])
retain_df = train_df[~train_df[ID_COL].isin(forget_ids)].reset_index(drop=True)

val_df = retain_df.sample(frac=0.10, random_state=SEED)
retain_fit_df = retain_df.drop(val_df.index).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"retain_fit={len(retain_fit_df)}  val={len(val_df)}  forget={len(forget_df)}")

def to_tensors(df):
    X = torch.tensor(df[FEATURE_COLS].fillna(0).values, dtype=torch.float32)
    y = torch.tensor(df[TARGET_COLS].fillna(0).values, dtype=torch.float32)
    return X, y

X_retain, y_retain = to_tensors(retain_fit_df)
X_val, y_val = to_tensors(val_df)
X_forget, y_forget = to_tensors(forget_df)

retain_loader = DataLoader(TensorDataset(X_retain, y_retain), batch_size=512, shuffle=True)
forget_loader = DataLoader(TensorDataset(X_forget, y_forget), batch_size=512, shuffle=True)


retain_fit=108628  val=12070  forget=9085


In [67]:
#evaluation functions
criterion = nn.BCEWithLogitsLoss()
criterion_none = nn.BCEWithLogitsLoss(reduction="none")

@torch.no_grad()
def per_sample_loss(m, X, y, bs=2048):
    m.eval(); out = []
    for i in range(0, len(X), bs):
        logits = m(X[i:i+bs].to(DEVICE))
        out.append(criterion_none(logits, y[i:i+bs].to(DEVICE)).mean(dim=1).cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def precision_at_k(m, X, y, k=10, bs=2048):
    m.eval(); hits, total = 0.0, 0
    for i in range(0, len(X), bs):
        logits = m(X[i:i+bs].to(DEVICE)).cpu()
        topk = logits.topk(k, dim=1).indices
        yb = y[i:i+bs]
        rel = yb.gather(1, topk).sum(dim=1)
        n_true = yb.sum(dim=1).clamp(max=k)
        mask = n_true > 0
        hits += (rel[mask] / n_true[mask]).sum().item()
        total += mask.sum().item()
    return hits / max(total, 1)

def mia_auc(m):
    lf = per_sample_loss(m, X_forget, y_forget)
    lv = per_sample_loss(m, X_val, y_val)
    labels = np.r_[np.ones_like(lf), np.zeros_like(lv)]
    return roc_auc_score(labels, np.r_[-lf, -lv])

def report(m, tag):
    p10, auc = precision_at_k(m, X_val, y_val), mia_auc(m)
    mia = 1 - 2 * abs(auc - 0.5)
    print(f"[{tag}] P@10={p10:.4f}  MIA_AUC={auc:.4f}  MIA_score={mia:.4f}  "
          f"combo={0.45*p10 + 0.45*mia:.4f}")

report(model, "original")

[original] P@10=0.6631  MIA_AUC=0.5012  MIA_score=0.9976  combo=0.7473


In [68]:
#netgrad+
def neggrad_plus(base, epochs=2, lr=1e-4, alpha=0.2):
    """Minimize L(retain) - alpha * L(forget).
    Descent on retain protects P@10; ascent on forget pushes its
    losses up toward 'never seen' levels. alpha controls the balance."""
    m = copy.deepcopy(base).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    m.train()
    for _ in range(epochs):
        f_iter = iter(forget_loader)
        for xb, yb in retain_loader:
            try:
                xf, yf = next(f_iter)
            except StopIteration:
                f_iter = iter(forget_loader); xf, yf = next(f_iter)
            opt.zero_grad()
            loss = (criterion(m(xb.to(DEVICE)), yb.to(DEVICE))
                    - alpha * criterion(m(xf.to(DEVICE)), yf.to(DEVICE)))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
    return m

**Improved version**

In [69]:
# alpha sweep with a fine-tuning grid that maximizes local val P under the constraint that local MIA_score stays ≥ 0.99

results = []
for lr in [1e-4, 3e-4, 1e-3]:
    for epochs in [2, 4]:
        torch.manual_seed(SEED)
        t0 = time.time()
        m = neggrad_plus(model, epochs=epochs, lr=lr, alpha=0.0)  # = pure retain fine-tune.. just to start
        t = int(time.time() - t0)
        p10 = precision_at_k(m, X_val, y_val)
        auc = mia_auc(m)
        mia = 1 - 2*abs(auc - 0.5)
        print(f"lr={lr} ep={epochs} time={t}s  P@10={p10:.4f}  MIA={mia:.4f}")
        results.append((p10, mia, lr, epochs, t, m))

lr=0.0001 ep=2 time=15s  P@10=0.6309  MIA=0.9961
lr=0.0001 ep=4 time=24s  P@10=0.6867  MIA=0.9959
lr=0.0003 ep=2 time=12s  P@10=0.7005  MIA=0.9964
lr=0.0003 ep=4 time=26s  P@10=0.7074  MIA=0.9963
lr=0.001 ep=2 time=12s  P@10=0.7085  MIA=0.9975
lr=0.001 ep=4 time=23s  P@10=0.6987  MIA=0.9956


In [70]:
# pick best P 10 among configs with MIA >= 0.99
valid = [r for r in results if r[1] >= 0.99]
best = max(valid, key=lambda r: r[0])
p10, mia, lr, epochs, t, final_model = best
exec_time = max(t, 1)
print(f"WINNER: lr={lr} epochs={epochs}  P@10={p10:.4f}  MIA={mia:.4f}  time={exec_time}s")

WINNER: lr=0.001 epochs=2  P@10=0.7085  MIA=0.9975  time=12s


In [71]:
SUBMISSIONS_DIR = Path("/content/hackathon-sapienza/submissions")

def make_submission(name, sd, exec_seconds):
    sub = SUBMISSIONS_DIR / name
    if sub.exists(): shutil.rmtree(sub)
    sub.mkdir(parents=True)

    payload = {
        "state_dict": {k: v.cpu() for k, v in sd.items()},
        "architecture": architecture,
        "best_hyperparameters": best_params,
        "model_class_source": model_class_source,
    }
    with open(sub / "model_artifact", "wb") as f:
        pickle.dump(payload, f)

    (sub / "execution_time.txt").write_text(str(int(exec_seconds)))

    val_df[[ID_COL]].rename(columns={ID_COL: "user_id"}).to_csv(
        sub / "validation_ids.csv", index=False)

    zip_path = SUBMISSIONS_DIR / f"{name}.zip"
    if zip_path.exists(): zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for fn in ["execution_time.txt", "model_artifact", "validation_ids.csv"]:
            z.write(sub / fn, arcname=f"{name}/{fn}")

    # verify
    loaded = pickle.load(open(sub / "model_artifact", "rb"))
    assert set(loaded) == {"state_dict", "architecture",
                           "best_hyperparameters", "model_class_source"}
    ids = pd.read_csv(sub / "validation_ids.csv")
    assert list(ids.columns) == ["user_id"] and not ids.user_id.duplicated().any()
    print(f"OK -> {zip_path}")
    return zip_path

In [72]:
GROUP = "itsTIMe"

zip_v0 = make_submission(f"{GROUP}_V0_laura", model.state_dict(), 1)
zip_ng = make_submission(f"{GROUP}_V2_laura", final_model.state_dict(), exec_time)

OK -> /content/hackathon-sapienza/submissions/itsTIMe_V0_laura.zip
OK -> /content/hackathon-sapienza/submissions/itsTIMe_V2_laura.zip


In [73]:

files.download(str(zip_v0))
files.download(str(zip_ng))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
def predicted_total(p10, mia, t):
    m3 = max(1 - 0.0084 * t, 0)          # linear fit from our 2 leaderboard points
    return 100 * (0.45 * p10 + 0.45 * mia + 0.10 * m3)

for lr in [1e-3, 3e-3]:
    for epochs in [1, 2, 4]:
        torch.manual_seed(SEED)
        t0 = time.time()
        m = neggrad_plus(model, epochs=epochs, lr=lr, alpha=0.0)
        t = max(int(time.time() - t0), 1)
        p10 = precision_at_k(m, X_val, y_val)
        mia = 1 - 2*abs(mia_auc(m) - 0.5)
        print(f"lr={lr} ep={epochs} t={t}s  P@10={p10:.4f}  MIA={mia:.4f}  pred_total={predicted_total(p10, mia, t):.2f}")
        results.append((p10, mia, lr, epochs, t, m))

lr=0.001 ep=1 t=6s  P@10=0.7020  MIA=0.9962  pred_total=85.91
lr=0.001 ep=2 t=12s  P@10=0.7085  MIA=0.9975  pred_total=85.76
lr=0.001 ep=4 t=24s  P@10=0.6987  MIA=0.9956  pred_total=84.23
lr=0.003 ep=1 t=6s  P@10=0.7061  MIA=0.9948  pred_total=86.04
lr=0.003 ep=2 t=14s  P@10=0.6994  MIA=0.9958  pred_total=85.11


In [ ]:
retain_loader_fast = DataLoader(TensorDataset(X_retain, y_retain), batch_size=4096, shuffle=True)

retain_loader_slow = retain_loader     # keep a handle
retain_loader = retain_loader_fast     # neggrad_plus reads the global
for lr in [1e-3, 3e-3]:
    torch.manual_seed(SEED)
    t0 = time.time()
    m = neggrad_plus(model, epochs=2, lr=lr, alpha=0.0)
    t = max(int(time.time() - t0), 1)
    p10 = precision_at_k(m, X_val, y_val)
    mia = 1 - 2*abs(mia_auc(m) - 0.5)
    print(f"BS4096 lr={lr} ep=2 t={t}s  P@10={p10:.4f}  MIA={mia:.4f}  pred_total={predicted_total(p10, mia, t):.2f}")
    results.append((p10, mia, lr, epochs, t, m))
retain_loader = retain_loader_slow     # restore

In [ ]:
best = max(results, key=lambda r: predicted_total(r[0], r[1], r[4]))
p10, mia, lr, epochs, t, final_model = best
print(f"V3 candidate: lr={lr} ep={epochs} t={t}s  P@10={p10:.4f}  MIA={mia:.4f}  pred={predicted_total(p10, mia, t):.2f}")
zip_v3 = make_submission(f"{GROUP}_VFINAL", final_model.state_dict(), t)

In [ ]:

files.download(str(zip_v3))